In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from data_tools.db import Manager

from mff.new_cap import generate_labels_from_bins

engine = Manager("mvh-admin", "mvh").engine


mpl.rcParams["figure.dpi"] = 200

In [ ]:
def read_data_tera_extended():
    with open("tera_allat.sql", "r", encoding="utf-8") as f:
        sql = f.read()

    data = pd.read_sql(sql, con=engine).fillna(0)

    bins = [0.1, 10, 150, 300, 1200, float("inf")]
    labels = generate_labels_from_bins(bins)

    data["area_class"] = pd.cut(
        data["area_biss_criss"], bins=bins, labels=labels, right=False
    )

    if "Nincs mezőgazdasági területe" not in data["area_class"].cat.categories:
        data["area_class"] = data["area_class"].cat.add_categories(
            ["Nincs mezőgazdasági területe"]
        )

    data.loc[data["area_biss_criss"].eq(0), "area_class"] = (
        "Nincs mezőgazdasági területe"
    )
    return data

In [ ]:
data = read_data_tera_extended()

In [ ]:
dict_param = {
    "tk_tejhasznu_tehen": "Termeléshez kötött támogatás - Tejhasznú tehénállomány",
    "tk_anyajuh": "Termeléshez kötött támogatás - anyajuh állomány",
    "tk_hizottbika": "Termeléshez kötött támogatás - hízottbika állomány",
    "tk_anyatehen": "Termeléshez kötött támogatás - anyatehén állomány",
    "tk_cukorrepa": "Termeléshez kötött támogatás - cukorrépa terület",
    "tk_ipari_zoldseg": "Termeléshez kötött támogatás - ipari zöldségnövény terület",
    "tk_zoldsegnoveny": "Termeléshez kötött támogatás - zöldségnövény terület",
    "tk_extenziv_gyumolcs": "Termeléshez kötött támogatás - extenzív gyümölcsterület",
    "tk_intenziv_gyumolcs": "Termeléshez kötött támogatás - intenzív gyümölcsterület",
    "tk_szemes_feherjenoveny": "Termeléshez kötött támogatás - szemes fehérjenövény terület",
}

param = "tk_intenziv_gyumolcs"
total = data[param].sum()
share = data.groupby("area_class", observed=False)[param].sum() / total * 100

plt.style.use("seaborn-v0_8-whitegrid")

fig, ax = plt.subplots(figsize=(8, 5))

colors = plt.cm.Blues(np.linspace(0.2, 0.9, len(share)))
bars = []
for i, (cat, v) in enumerate(zip(share.index, share.values)):
    if cat == "Nincs mezőgazdasági területe":
        bar = ax.bar(
            cat,
            v,
            color="lightgrey",
            edgecolor="black",
            hatch="//",  # csíkozás
        )
    else:
        bar = ax.bar(
            cat,
            v,
            color=colors[i],
            edgecolor="black",
        )
    bars.append(bar[0])


ax.set_ylabel("Arány (%)")
ax.set_title(
    f"{dict_param[param]} százalékos megoszlása\nbirtokméret kategóriák szerint"
)
plt.xticks(rotation=45)

for bar, v in zip(bars, share.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{v:.1f}%",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold",
    )

plt.tight_layout()
plt.show()

In [ ]:
dict_param = {
    "tk_tejhasznu_tehen": "Termeléshez kötött támogatás - Tejhasznú igénylők",
    "tk_anyajuh": "Termeléshez kötött támogatás - anyajuh igénylők",
    "tk_hizottbika": "Termeléshez kötött támogatás - hízottbika igénylők",
    "tk_anyatehen": "Termeléshez kötött támogatás - anyatehén igénylők",
    "tk_cukorrepa": "Termeléshez kötött támogatás - cukorrépa igénylők",
    "tk_ipari_zoldseg": "Termeléshez kötött támogatás - ipari zöldségnövény igénylők",
    "tk_zoldsegnoveny": "Termeléshez kötött támogatás - zöldségnövény igénylők",
    "tk_extenziv_gyumolcs": "Termeléshez kötött támogatás - extenzív gyümölcs igénylők",
    "tk_intenziv_gyumolcs": "Termeléshez kötött támogatás - intenzív gyümölcs igénylők",
    "tk_szemes_feherjenoveny": "Termeléshez kötött támogatás - szemes fehérjenövény igénylők",
}

param = "tk_szemes_feherjenoveny"

mask = data[param] > 0
share = data[mask].groupby("area_class", observed=False).size() / mask.sum() * 100

plt.style.use("seaborn-v0_8-whitegrid")

fig, ax = plt.subplots(figsize=(8, 5))

colors = plt.cm.Blues(np.linspace(0.2, 0.9, len(share)))
bars = []
for i, (cat, v) in enumerate(zip(share.index, share.values)):
    if cat == "Nincs mezőgazdasági területe":
        bar = ax.bar(
            cat,
            v,
            color="lightgrey",
            edgecolor="black",
            hatch="//",  # csíkozás
        )
    else:
        bar = ax.bar(
            cat,
            v,
            color=colors[i],
            edgecolor="black",
        )
    bars.append(bar[0])


ax.set_ylabel("Üzemszám arány (%)")
ax.set_title(
    f"{dict_param[param]} százalékos megoszlása\nbirtokméret kategóriák szerint"
)
plt.xticks(rotation=45)

for bar, v in zip(bars, share.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{v:.1f}%",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold",
    )

plt.tight_layout()
plt.show()

In [ ]:
df = data[(data["area_biss_criss"] <= 20_000) & (data["tk_szemes_feherjenoveny"] > 0)]

X = df["area_biss_criss"]
y = df["tk_szemes_feherjenoveny"]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def scatter_with_ols(x, y, title="Kapcsolat a BISS+CRISS terület és TK között"):
    # --- Fit OLS line ---
    m, b = np.polyfit(x, y, 1)
    yhat = m * x + b
    ss_res = np.sum((y - yhat) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    r2 = 1 - ss_res / ss_tot

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(8, 6), dpi=130)

    # scatter: smaller points, alpha for density
    ax.scatter(
        x, y, s=14, alpha=0.4, color="tab:blue", edgecolor="none", label="Adatok"
    )

    # OLS line
    xx = np.linspace(np.min(x), np.max(x), 200)
    ax.plot(xx, m * xx + b, color="tab:red", lw=2.2, label="OLS illesztés")

    # equation box (top-left corner)
    eqn = f"$y = {b:.2f} + {m:.2f}x$\n$R^2 = {r2:.3f}$"
    ax.text(
        0.02,
        0.95,
        eqn,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=10,
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="0.8"),
    )

    # Labels, title, style
    ax.set_title(title, fontsize=13, pad=12)
    ax.set_xlabel("BISS+CRISS terület (ha)")
    ax.set_ylabel("TK")
    ax.grid(True, which="major", linestyle="--", linewidth=0.6, alpha=0.6)

    # cleaner frame
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

    # Legend
    ax.legend(frameon=False, loc="upper right")

    fig.tight_layout()
    return fig, ax


fig, ax = scatter_with_ols(X, y)
plt.show()

In [ ]:
X_const = sm.add_constant(X)

model = sm.OLS(y, X_const).fit()
print(model.summary())

x_pred = pd.Series(sorted(X))
X_pred = sm.add_constant(x_pred)
y_pred = model.predict(X_pred)

intercept, slope = model.params
r2 = model.rsquared

plt.figure(figsize=(6, 6))
plt.scatter(X, y, alpha=0.6, label="Adatok")
plt.plot(x_pred, y_pred, color="red", linewidth=2, label="OLS illesztés")

eq_text = f"y = {intercept:.2f} + {slope:.2f}x\n$R^2$ = {r2:.3f}"
plt.text(
    0.05,
    0.95,
    eq_text,
    transform=plt.gca().transAxes,
    fontsize=10,
    verticalalignment="top",
    bbox=dict(facecolor="white", alpha=0.7, edgecolor="grey"),
)

plt.xlabel("Area BISS+CRISS")
plt.ylabel("TK terület")
plt.title("Kapcsolat a BISS+CRISS terület és TK között")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
df = data[(data["area_biss_criss"] <= 1_000) & (data["tk_szemes_feherjenoveny"] > 0)]

# 1) Egy magyarázó változó 2D exog formában, konstans nélkül
X = df[["area_biss_criss"]]  # <- NINCS add_constant
y = df["tk_szemes_feherjenoveny"]

# 2) OLS 0 intercept-tel (alapesetben nincs konstans, tehát y = b * x)
model = sm.OLS(y, X).fit()
print(model.summary())

# 3) Előrejelzéshez is 2D DataFrame kell ugyanazzal az oszlopnévvel
x_pred = df["area_biss_criss"].sort_values()
X_pred = pd.DataFrame({"area_biss_criss": x_pred.to_numpy()})
y_pred = model.predict(X_pred)

# 4) Paraméterek és R^2 (0-intercept modellnél TSS = sum(y^2))
slope = model.params["area_biss_criss"]
r2 = model.rsquared

# 5) Ábra
plt.figure(figsize=(6, 6))
plt.scatter(X["area_biss_criss"], y, alpha=0.6, label="Adatok")
plt.plot(x_pred, y_pred, linewidth=2, label="OLS illesztés (0 metszet)")

eq_text = f"y = {slope:.2f} · x\n$R^2$ = {r2:.3f}"
plt.text(
    0.05,
    0.95,
    eq_text,
    transform=plt.gca().transAxes,
    fontsize=10,
    va="top",
    bbox=dict(facecolor="white", alpha=0.7, edgecolor="grey"),
)

plt.xlabel("Area BISS+CRISS")
plt.ylabel("TK terület")
plt.title("Kapcsolat a BISS+CRISS terület és TK között (0 intercept)")
plt.grid(True)
plt.legend()
plt.show()